# 03 LSH Model Construction with MinHash

This notebook builds the Locality Sensitive Hashing recommender on top of preprocessed movie tags.

Note: This is the production-serving path used by the Flask app and default training pipeline.
Cosine-based comparison is kept separately in Notebook 04 for offline analysis only.

## What this notebook does
- Loads the cleaned dataset from Notebook 01.
- Builds MinHash signatures for each movie tag string.
- Indexes signatures with MinHashLSH for fast approximate nearest-neighbor search.
- Validates recommendations for a sample title.
- Saves reusable LSH artifacts for the Flask app and Notebook 04 analysis.

## Expected artifacts
- models/artifacts/lsh_index.pkl
- models/artifacts/minhash_store.pkl

### 1) Import libraries

In [5]:
import pickle
from pathlib import Path

import pandas as pd
from datasketch import MinHash, MinHashLSH

### 2) Load processed data

In [6]:
processed_data_path = Path("../data/processed_movies.csv")
movies_df = pd.read_csv(processed_data_path)

In [7]:
id_col = "tmdb_id" if "tmdb_id" in movies_df.columns else "movie_id"
required_columns = {id_col, "title", "tags"}

In [8]:
missing = required_columns.difference(movies_df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

In [9]:
movies_df = movies_df.dropna(subset=[id_col, "title", "tags"]).copy()
movies_df = movies_df.drop_duplicates(subset=[id_col]).reset_index(drop=True)
movies_df["tags"] = movies_df["tags"].astype(str).str.strip().str.lower()
movies_df = movies_df[movies_df["tags"].str.len() > 0].reset_index(drop=True)

In [10]:
print("Shape:", movies_df.shape)
print(movies_df.head(3))

Shape: (4800, 8)
   tmdb_id                                     title  \
0    19995                                    Avatar   
1      285  Pirates of the Caribbean: At World's End   
2   206647                                   Spectre   

                                            overview  \
0  In the 22nd century, a paraplegic Marine is di...   
1  Captain Barbossa, long believed to be dead, ha...   
2  A cryptic message from Bond’s past sends him o...   

                                                tags  poster_path  \
0  in the 22nd century, a paraplegic marine is di...          NaN   
1  captain barbossa, long believed to be dead, ha...          NaN   
2  a cryptic message from bond’s past sends him o...          NaN   

  release_date  vote_average  vote_count  
0   2009-12-10           7.2       11800  
1   2007-05-19           6.9        4500  
2   2015-10-26           6.3        4466  


## Define MinHash and LSH Helpers

The next cells create reusable functions for signature generation, index construction, and recommendation retrieval.

### 3) Create helper functions for MinHash and LSH

In [ ]:
MINHASH_NUM_PERM = 96
LSH_THRESHOLD = 0.2

In [12]:
def build_minhash_signature(tokens, num_perm=MINHASH_NUM_PERM):
    signature = MinHash(num_perm=num_perm)
    for token in sorted(set(tokens)):
        signature.update(token.encode("utf8"))
    return signature


def build_lsh_index(dataframe, num_perm=MINHASH_NUM_PERM, threshold=LSH_THRESHOLD):
    lsh_index = MinHashLSH(threshold=threshold, num_perm=num_perm)
    minhash_store = {}

    for row_idx, tag_text in enumerate(dataframe["tags"]):
        movie_key = str(row_idx)
        tokens = str(tag_text).split()
        signature = build_minhash_signature(tokens, num_perm=num_perm)

        lsh_index.insert(movie_key, signature)
        minhash_store[movie_key] = signature

    return lsh_index, minhash_store

### 4) Build LSH index

In [13]:
lsh_index, minhash_store = build_lsh_index(movies_df)

print("LSH index built successfully.")
print("Total movie signatures:", len(minhash_store))
print("Configured num_perm:", MINHASH_NUM_PERM)
print("Configured threshold:", LSH_THRESHOLD)

LSH index built successfully.
Total movie signatures: 4800
Configured num_perm: 128
Configured threshold: 0.2


### 5) LSH recommendation function

In [14]:
def recommend_lsh(movie_name, dataframe, lsh_obj, minhash_dict, top_n=5):
    title_series = dataframe["title"].astype(str).str.lower().str.strip()
    target = movie_name.lower().strip()

    exact_matches = dataframe.index[title_series == target]
    if len(exact_matches) > 0:
        movie_index = int(exact_matches[0])
    else:
        contains_matches = dataframe.index[title_series.str.contains(target, regex=False)]
        if len(contains_matches) == 0:
            raise ValueError(f"Movie '{movie_name}' not found.")
        movie_index = int(contains_matches[0])

    query_key = str(movie_index)
    query_signature = minhash_dict[query_key]

    candidate_keys = lsh_obj.query(query_signature)
    candidate_indices = []
    for key in candidate_keys:
        try:
            idx = int(str(key))
        except (TypeError, ValueError):
            continue
        if idx != movie_index:
            candidate_indices.append(idx)

    if len(candidate_indices) < top_n:
        candidate_indices = [idx for idx in range(len(dataframe)) if idx != movie_index]

    scored = []
    for idx in candidate_indices:
        score = query_signature.jaccard(minhash_dict[str(idx)])
        scored.append((idx, score))

    scored.sort(key=lambda item: item[1], reverse=True)
    top_scored = scored[:top_n]

    top_indices = [idx for idx, _ in top_scored]
    top_scores = [score for _, score in top_scored]

    rec_df = dataframe.iloc[top_indices][[id_col, "title"]].copy()
    rec_df["score"] = top_scores

    return rec_df.reset_index(drop=True)

### 6) Quick LSH recommendation test

In [15]:
sample_movie = "Avatar"
lsh_recommendations = recommend_lsh(sample_movie, movies_df, lsh_index, minhash_store, top_n=5)

print(f"Top 5 LSH recommendations for '{sample_movie}':")
print(lsh_recommendations)

Top 5 LSH recommendations for 'Avatar':
   tmdb_id                              title     score
0    23531                            Whipped  0.148438
1    76757                  Jupiter Ascending  0.148438
2   228326                   The Book of Life  0.140625
3    36670              Never Say Never Again  0.140625
4    28121  A Thin Line Between Love and Hate  0.132812


## Persist and Validate Artifacts

Save the LSH objects to disk, then reload them to confirm serialization integrity before deployment use.

### 7) Save artifacts

In [16]:
artifacts_dir = Path("../models/artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)

In [17]:
lsh_index_path = artifacts_dir / "lsh_index.pkl"
minhash_store_path = artifacts_dir / "minhash_store.pkl"

In [18]:
with open(lsh_index_path, "wb") as file:
    pickle.dump(lsh_index, file)

with open(minhash_store_path, "wb") as file:
    pickle.dump(minhash_store, file)

In [19]:
print("Saved artifacts:")
print("-", lsh_index_path)
print("-", minhash_store_path)

Saved artifacts:
- ..\models\artifacts\lsh_index.pkl
- ..\models\artifacts\minhash_store.pkl


### 8) Validate artifact loading

In [20]:
with open(lsh_index_path, "rb") as file:
    loaded_lsh_index = pickle.load(file)

with open(minhash_store_path, "rb") as file:
    loaded_minhash_store = pickle.load(file)

In [21]:
print("Loaded LSH index type:", type(loaded_lsh_index))
print("Loaded MinHash store size:", len(loaded_minhash_store))

Loaded LSH index type: <class 'datasketch.lsh.MinHashLSH'>
Loaded MinHash store size: 4800
